# Flutter’ın Klavye Odak Sistemini Anlamak

Flutter uygulamanızda odak sistemini nasıl kullanacağınız hakkında
bilgiler.

Bu makale, klavye girişinin nereye yönlendirileceğini nasıl kontrol
edeceğinizi açıklar. Çoğu masaüstü ve web uygulaması gibi fiziksel bir
klavye kullanan bir uygulama geliştiriyorsanız, bu sayfa tam size göre.
Eğer uygulamanız fiziksel bir klavye ile kullanılmayacaksa bu kısmı
atlayabilirsiniz.

## Genel Bakış

Flutter, klavye girişini uygulamanın belirli bir bölümüne yönlendiren
bir odak (focus) sistemi ile birlikte gelir. Bunu yapmak için
kullanıcılar, istedikleri UI öğesine dokunarak veya tıklayarak girişi
uygulamanın o kısmına “odaklar”. Bu gerçekleştiğinde, klavyeyle girilen
metin, odak uygulamanın başka bir kısmına taşınana kadar o kısma akar.
Odak, genellikle `Tab` tuşuna bağlı olan belirli bir klavye kısayoluyla
da taşınabilir; buna bazen “tab geçişi” (tab traversal) denir.

Bu sayfa, bir Flutter uygulamasında bu işlemleri gerçekleştirmek için
kullanılan API’leri ve odak sisteminin nasıl çalıştığını inceler.
Geliştiriciler arasında `FocusNode` nesnelerinin nasıl tanımlanacağı ve
kullanılacağı konusunda bazı kafa karışıklıkları olduğunu fark ettik.
Eğer bu durum sizi tarif ediyorsa, doğrudan `FocusNode` nesneleri
oluşturmak için en iyi uygulamalar bölümüne geçebilirsiniz.

### Odak Kullanım Senaryoları

Odak sistemini nasıl kullanacağınızı bilmeniz gereken durumlara bazı
örnekler: \* Tuş olaylarını (key events) alma/işleme. \* Odaklanabilir
olması gereken özel bir bileşen uygulama. \* Odak değiştiğinde bildirim
alma. \* Uygulamadaki odak geçişinin “tab sırasını” değiştirme veya
tanımlama. \* Birlikte gezilmesi gereken kontrol gruplarını tanımlama.
\* Uygulamadaki bazı kontrollerin odaklanabilir olmasını engelleme.

### Sözlük

Aşağıda, Flutter’ın odak sisteminin öğeleri için kullandığı terimler yer
almaktadır. Bu kavramların bazılarını uygulayan çeşitli sınıflar aşağıda
tanıtılmıştır.

- **Odak ağacı (Focus tree):** Tipik olarak widget ağacını seyrek bir
  şekilde yansıtan ve odak alabilen tüm widget’ları temsil eden odak
  düğümleri ağacı.
- **Odak düğümü (Focus node):** Odak ağacındaki tek bir düğüm. Bu düğüm
  odağı alabilir ve odak zincirinin bir parçası olduğunda “odağa sahip”
  olduğu söylenir. Yalnızca odağa sahip olduğunda tuş olaylarını
  işlemeye katılır.
- **Birincil odak (Primary focus):** Odak ağacının kökünden en uzaktaki,
  odağa sahip olan odak düğümü. Tuş olaylarının yayılmaya (propagate)
  başladığı düğüm burasıdır.
- **Odak zinciri (Focus chain):** Birincil odak düğümünden başlayan ve
  odak ağacının dallarını takip ederek köke kadar uzanan sıralı odak
  düğümleri listesi.
- **Odak kapsamı (Focus scope):** Görevi diğer odak düğümlerinden oluşan
  bir grubu barındırmak ve yalnızca bu düğümlerin odak almasına izin
  vermek olan özel bir odak düğümü. Alt ağacında daha önce hangi
  düğümlerin odaklandığına dair bilgileri tutar.
- **Odak geçişi (Focus traversal):** Odaklanabilir bir düğümden diğerine
  öngörülebilir bir sırayla geçme işlemi. Bu genellikle uygulamalarda
  kullanıcının bir sonraki odaklanabilir kontrole veya alana geçmek için
  `Tab` tuşuna basmasıyla görülür.

## FocusNode ve FocusScopeNode

`FocusNode` ve `FocusScopeNode` nesneleri odak sisteminin mekaniğini
uygular. Bunlar, widget ağacının derlemeleri (builds) arasında kalıcı
olmaları için odak durumunu ve niteliklerini tutan uzun ömürlü
nesnelerdir (widget’lardan daha uzun ömürlü, render nesnelerine benzer).
Birlikte, odak ağacı veri yapısını oluştururlar.

Başlangıçta odak sisteminin bazı yönlerini kontrol etmek için
geliştiricilere yönelik nesneler olmaları amaçlanmıştı, ancak zamanla
çoğunlukla odak sisteminin ayrıntılarını uygulamaya evrildiler. Mevcut
uygulamaları bozmamak için, nitelikleri için hala genel arayüzler
içerirler. Ancak genel olarak en yararlı oldukları nokta; bir üst
(ancestor) widget’ta `requestFocus()` çağırmak ve böylece bir alt
(descendant) widget’ın odak almasını talep etmek için, alt widget’a
aktarılan nispeten opak bir tutamaç (handle) görevi görmeleridir. Diğer
niteliklerin ayarlanması, bunları kullanmıyorsanız veya kendi
versiyonunuzu uygulamıyorsanız, en iyi `Focus` veya `FocusScope`
widget’ı tarafından yönetilir.

### FocusNode Nesneleri Oluşturmak İçin En İyi Uygulamalar

Bu nesneleri kullanırken yapılması ve yapılmaması gerekenler:

- **Yapmayın:** Her derleme (build) için yeni bir `FocusNode` tahsis
  etmeyin. Bu, bellek sızıntılarına neden olabilir ve bazen düğüm
  odaktayken widget yeniden oluşturulduğunda odağın kaybolmasına neden
  olur.
- **Yapın:** `FocusNode` ve `FocusScopeNode` nesnelerini bir
  `StatefulWidget` içinde oluşturun. `FocusNode` ve `FocusScopeNode`,
  kullanımları bittiğinde imha edilmelidir (dispose), bu nedenle
  yalnızca `dispose` yöntemini geçersiz kılarak (override) onları
  temizleyebileceğiniz bir stateful widget’ın durum nesnesi içinde
  oluşturulmalıdırlar.
- **Yapmayın:** Aynı `FocusNode`’u birden fazla widget için kullanmayın.
  Bunu yaparsanız, widget’lar düğümün niteliklerini yönetmek için
  birbiriyle çatışır ve muhtemelen beklediğiniz sonucu alamazsınız.
- **Yapın:** Odak sorunlarını teşhis etmeye yardımcı olması için odak
  düğümü widget’ının `debugLabel` özelliğini ayarlayın.
- **Yapmayın:** Eğer `Focus` veya `FocusScope` widget’ı tarafından
  yönetiliyorlarsa, `FocusNode` veya `FocusScopeNode` üzerinde
  `onKeyEvent` geri çağrısını ayarlamayın. Bir `onKeyEvent` işleyicisi
  istiyorsanız, dinlemek istediğiniz widget alt ağacının etrafına yeni
  bir `Focus` widget’ı ekleyin ve widget’ın `onKeyEvent` niteliğini
  işleyicinize ayarlayın. Ayrıca birincil odağı almasını istemiyorsanız
  widget üzerinde `canRequestFocus: false` olarak ayarlayın. Bunun
  nedeni, `Focus` widget’ındaki `onKeyEvent` niteliğinin sonraki bir
  derlemede başka bir şeye ayarlanabilmesi ve bu gerçekleşirse, düğüm
  üzerinde ayarladığınız `onKeyEvent` işleyicisinin üzerine
  yazılmasıdır.
- **Yapın:** Özellikle odaklanmasını istediğiniz bir alt öğeye sahip
  olduğunuz bir üst öğeden, o düğümün birincil odağı almasını talep
  etmek için düğüm üzerinde `requestFocus()` çağırın.
- **Yapın:** `focusNode.requestFocus()` kullanın.
  `FocusScope.of(context).requestFocus(focusNode)` çağırmak gerekli
  değildir. `focusNode.requestFocus()` yöntemi eşdeğerdir ve daha
  performanslıdır.

### Odaktan Çıkma (Unfocusing)

Bir düğüme “odağı bırakmasını” söylemek için `FocusNode.unfocus()`
adında bir API vardır. Bu, odağı düğümden kaldırsa da, aslında tüm
düğümlerin “odaktan çıkması” gibi bir durumun olmadığını anlamak
önemlidir. Bir düğüm odaktan çıkarsa, odağı başka bir yere aktarmalıdır
çünkü **her zaman** bir birincil odak vardır. Bir düğüm `unfocus()`
çağırdığında odağı alan düğüm, `unfocus()`’a verilen `disposition`
argümanına bağlı olarak ya en yakın `FocusScopeNode` ya da o kapsamda
daha önce odaklanmış bir düğümdür. Odağı bir düğümden kaldırdığınızda
nereye gideceği üzerinde daha fazla kontrol istiyorsanız, `unfocus()`
çağırmak yerine açıkça başka bir düğümü odaklayın veya `FocusNode`
üzerindeki `focusInDirection`, `nextFocus` veya `previousFocus`
yöntemleriyle odak geçiş mekanizmasını kullanarak başka bir düğüm bulun.

`unfocus()` çağrıldığında, `disposition` argümanı odaktan çıkma için iki
moda izin verir: `UnfocusDisposition.scope` ve
`UnfocusDisposition.previouslyFocusedChild`. Varsayılan `scope`’tur, bu
da odağı en yakın ebeveyn odak kapsamına verir. Bu, odağın daha sonra
`FocusNode.nextFocus` ile bir sonraki düğüme taşınması durumunda,
kapsamdaki “ilk” odaklanabilir öğeden başlayacağı anlamına gelir.

`previouslyFocusedChild` düzenlemesi (disposition), daha önce odaklanmış
çocuğu bulmak için kapsamı arar ve onun odaklanmasını talep eder. Daha
önce odaklanmış bir çocuk yoksa, `scope` ile eşdeğerdir.

#### Dikkat

Başka bir kapsam yoksa, odak sisteminin kök kapsam düğümü olan
`FocusManager.rootScope`’a geçer. Kök kapsamın, çerçevenin bir sonraki
hangi düğümün odaklanması gerektiğini belirlemesi için bir `context`’i
olmadığından bu genellikle istenmeyen bir durumdur. Uygulamanızın aniden
odak geçişini (tab tuşu vb.) kullanarak gezinme yeteneğini kaybettiğini
fark ederseniz, muhtemelen olan budur. Düzeltmek için, odaktan çıkmayı
talep eden odak düğümüne bir ata (ancestor) olarak `FocusScope` ekleyin.
`WidgetsApp` (ki `MaterialApp` ve `CupertinoApp` bundan türetilmiştir)
kendi `FocusScope`’una sahiptir, bu nedenle bunları kullanıyorsanız bu
bir sorun olmamalıdır.

## Focus Widget

`Focus` widget’ı bir odak düğümüne sahiptir ve onu yönetir; odak
sisteminin iş yükünü çeken ana parçasıdır. Sahip olduğu odak düğümünün
odak ağacına eklenmesini ve çıkarılmasını yönetir, odak düğümünün
niteliklerini ve geri çağrılarını (callbacks) yönetir ve widget ağacına
bağlı odak düğümlerinin keşfedilmesini sağlayan statik işlevlere
sahiptir.

En basit haliyle, bir widget alt ağacını `Focus` widget’ı ile
sarmalamak, o widget alt ağacının odak geçiş işleminin bir parçası
olarak veya kendisine iletilen `FocusNode` üzerinde `requestFocus`
çağrıldığında odak almasına olanak tanır. `requestFocus` çağıran bir
jest algılayıcı (gesture detector) ile birleştirildiğinde,
dokunulduğunda veya tıklandığında odak alabilir.

Yönetmesi için `Focus` widget’ına bir `FocusNode` nesnesi
geçirebilirsiniz, ancak geçirmezseniz o kendi nesnesini oluşturur. Kendi
`FocusNode`’unuzu oluşturmanın ana nedeni, odağı bir ebeveyn widget’tan
kontrol etmek için düğüm üzerinde `requestFocus()` çağırabilmektir. Bir
`FocusNode`’un diğer işlevlerinin çoğuna en iyi `Focus` widget’ının
kendi niteliklerini değiştirerek erişilir.

`Focus` widget’ı, odak işlevlerini uygulamak için Flutter’ın kendi
kontrollerinin çoğunda kullanılır.

İşte özel bir kontrolü odaklanabilir hale getirmek için `Focus`
widget’ının nasıl kullanılacağını gösteren bir örnek. Odak alındığında
tepki veren metin içeren bir kapsayıcı (container) oluşturur.

``` dart
import 'package:flutter/material.dart';

void main() => runApp(const MyApp());

class MyApp extends StatelessWidget {
  const MyApp({super.key});
  static const String _title = 'Focus Sample';

  @override
  Widget build(BuildContext context) {
    return MaterialApp(
      title: _title,
      home: Scaffold(
        appBar: AppBar(title: const Text(_title)),
        body: const Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: <Widget>[MyCustomWidget(), MyCustomWidget()],
        ),
      ),
    );
  }
}

class MyCustomWidget extends StatefulWidget {
  const MyCustomWidget({super.key});

  @override
  State<MyCustomWidget> createState() => _MyCustomWidgetState();
}

class _MyCustomWidgetState extends State<MyCustomWidget> {
  Color _color = Colors.white;
  String _label = 'Unfocused';

  @override
  Widget build(BuildContext context) {
    return Focus(
      onFocusChange: (focused) {
        setState(() {
          _color = focused ? Colors.black26 : Colors.white;
          _label = focused ? 'Focused' : 'Unfocused';
        });
      },
      child: Center(
        child: Container(
          width: 300,
          height: 50,
          alignment: Alignment.center,
          color: _color,
          child: Text(_label),
        ),
      ),
    );
  }
}
```

### Tuş Olayları (Key Events)

Bir alt ağaçtaki tuş olaylarını dinlemek istiyorsanız, `Focus`
widget’ının `onKeyEvent` niteliğini, yalnızca tuşu dinleyen veya tuşu
işleyip diğer widget’lara yayılmasını durduran bir işleyiciye ayarlayın.

Tuş olayları, birincil odağa sahip odak düğümünde başlar. Eğer bu düğüm
`onKeyEvent` işleyicisinden `KeyEventResult.handled` döndürmezse, olay
ebeveyn odak düğümüne verilir. Ebeveyn de işlemezse, kendi ebeveynine
gider ve bu böylece odak ağacının köküne kadar devam eder. Olay
işlenmeden odak ağacının köküne ulaşırsa, uygulamadaki bir sonraki yerel
(native) kontrole verilmek üzere platforma geri döndürülür (Flutter
UI’nın daha büyük bir yerel uygulama UI’sının parçası olması durumunda).
İşlenen olaylar diğer Flutter widget’larına ve yerel widget’lara
yayılmaz.

İşte alt ağacının işlemediği her tuşu absorbe eden (yutan), ancak
kendisi birincil odak olamayan bir `Focus` widget örneği:

``` dart
@override
Widget build(BuildContext context) {
  return Focus(
    onKeyEvent: (node, event) => KeyEventResult.handled,
    canRequestFocus: false, // Odak talep edemez
    child: child,
  );
}
```

Odak tuş olayları, metin giriş olaylarından önce işlenir; bu nedenle bir
metin alanını çevreleyen odak widget’ı bir tuş olayını işlediğinde, o
tuşun metin alanına girilmesi engellenir.

İşte “a” harfinin metin alanına yazılmasına izin vermeyen bir widget
örneği:

``` dart
@override
Widget build(BuildContext context) {
  return Focus(
    onKeyEvent: (node, event) {
      return (event.logicalKey == LogicalKeyboardKey.keyA)
          ? KeyEventResult.handled
          : KeyEventResult.ignored;
    },
    child: const TextField(),
  );
}
```

Amaç giriş doğrulaması ise, bu örneğin işlevselliği muhtemelen bir
`TextInputFormatter` kullanılarak daha iyi uygulanır, ancak teknik yine
de yararlı olabilir: örneğin `Shortcuts` widget’ı, kısayolları metin
girişi haline gelmeden önce işlemek için bu yöntemi kullanır.

### Neyin Odak Alacağını Kontrol Etmek

Odağın temel yönlerinden biri, neyin odak alabileceğini ve nasıl
alabileceğini kontrol etmektir. `canRequestFocus`, `skipTraversal` ve
`descendantsAreFocusable` nitelikleri, bu düğümün ve alt öğelerinin odak
sürecine nasıl katılacağını kontrol eder.

Eğer `skipTraversal` niteliği doğru (true) ise, bu odak düğümü odak
geçişine katılmaz. Odak düğümünde `requestFocus` çağrılırsa hala
odaklanabilir, ancak odak geçiş sistemi odaklanacak bir sonraki şeyi
ararken atlanır.

`canRequestFocus` niteliği, beklendiği gibi, bu `Focus` widget’ının
yönettiği odak düğümünün odak talep etmek için kullanılıp
kullanılamayacağını kontrol eder. Bu nitelik yanlış (false) ise, düğüm
üzerinde `requestFocus` çağırmanın hiçbir etkisi olmaz. Ayrıca odak
talep edemediği için bu düğümün odak geçişi sırasında atlanacağı
anlamına da gelir.

`descendantsAreFocusable` niteliği, bu düğümün alt öğelerinin odak alıp
alamayacağını kontrol eder, ancak yine de bu düğümün odak almasına izin
verir. Bu nitelik, tüm bir widget alt ağacı için odaklanabilirliği
kapatmak amacıyla kullanılabilir. `ExcludeFocus` widget’ı bu şekilde
çalışır: sadece bu niteliği ayarlanmış bir `Focus` widget’ıdır.

### Otomatik Odaklanma (Autofocus)

Bir `Focus` widget’ının `autofocus` niteliğini ayarlamak, widget’a ait
olduğu odak kapsamı ilk kez odaklandığında odak talep etmesini söyler.
Birden fazla widget’ta `autofocus` ayarlanmışsa, hangisinin odak alacağı
keyfidir, bu nedenle odak kapsamı başına yalnızca bir widget üzerinde
ayarlamaya çalışın.

`autofocus` niteliği, yalnızca düğümün ait olduğu kapsamda zaten bir
odak yoksa etkili olur.

Farklı odak kapsamlarına ait iki düğüm üzerinde `autofocus` niteliğini
ayarlamak iyi tanımlanmıştır: her biri, karşılık gelen kapsamları
odaklandığında odaklanmış widget olur.

### Değişiklik Bildirimleri

`Focus.onFocusChanged` geri çağrısı, belirli bir düğüm için odak
durumunun değiştiğine dair bildirimler almak için kullanılabilir. Düğüm
odak zincirine eklendiğinde veya zincirden çıkarıldığında bildirir, bu
da birincil odak olmasa bile bildirim aldığı anlamına gelir. Yalnızca
birincil odağı alıp almadığınızı bilmek istiyorsanız, odak düğümü
üzerinde `hasPrimaryFocus` değerinin doğru olup olmadığını kontrol edin.

### FocusNode’u Elde Etmek

Bazen, niteliklerini sorgulamak için bir `Focus` widget’ının odak
düğümünü elde etmek yararlıdır.

Odak düğümüne `Focus` widget’ının bir üst öğesinden (ancestor) erişmek
için, bir `FocusNode` oluşturun ve `Focus` widget’ının `focusNode`
niteliği olarak geçirin. İmha edilmesi gerektiğinden, geçirdiğiniz odak
düğümünün bir stateful widget’a ait olması gerekir, bu yüzden her
derlemede (build) yeni bir tane oluşturmayın.

`Focus` widget’ının bir alt öğesinden (descendant) odak düğümüne
erişmeniz gerekirse, verilen bağlama (context) en yakın `Focus`
widget’ının odak düğümünü elde etmek için `Focus.of(context)` çağrısı
yapabilirsiniz. Aynı `build` işlevi içindeki bir `Focus` widget’ının
`FocusNode`’unu elde etmeniz gerekirse, doğru bağlama sahip olduğunuzdan
emin olmak için bir `Builder` kullanın. Bu, aşağıdaki örnekte
gösterilmiştir:

``` dart
@override
Widget build(BuildContext context) {
  return Focus(
    child: Builder(
      builder: (context) {
        final bool hasPrimary = Focus.of(context).hasPrimaryFocus;
        print('Birincil odak ile oluşturuluyor: $hasPrimary');
        return const SizedBox(width: 100, height: 100);
      },
    ),
  );
}
```

### Zamanlama

Odak sisteminin ayrıntılarından biri, odak talep edildiğinde bunun
yalnızca mevcut derleme (build) aşaması tamamlandıktan sonra etkili
olmasıdır. Bu, odak değişikliklerinin her zaman bir kare (frame)
gecikmeli olduğu anlamına gelir, çünkü odağı değiştirmek, o anda odak
talep eden widget’ın üst öğeleri de dahil olmak üzere widget ağacının
keyfi bölümlerinin yeniden oluşturulmasına neden olabilir. Alt öğeler
üst öğelerini kirletemeyeceğinden (dirty), gerekli değişikliklerin bir
sonraki karede gerçekleşebilmesi için bu işlemin kareler arasında olması
gerekir.

## FocusScope Widget

`FocusScope` widget’ı, `FocusNode` yerine `FocusScopeNode` yöneten
`Focus` widget’ının özel bir sürümüdür. `FocusScopeNode`, bir alt
ağaçtaki odak düğümleri için bir gruplama mekanizması görevi gören odak
ağacındaki özel bir düğümdür. Kapsam dışındaki bir düğüm açıkça
odaklanmadıkça, odak geçişi bir odak kapsamı içinde kalır.

Odak kapsamı ayrıca geçerli odağı ve alt ağacında odaklanan düğümlerin
geçmişini de takip eder. Bu sayede, bir düğüm odağı bıraktığında veya
odaktayken kaldırıldığında, odak daha önce odağa sahip olan düğüme geri
döndürülebilir.

Odak kapsamları ayrıca, alt öğelerin hiçbirinde odak yoksa odağın
döndürüleceği bir yer görevi görür. Bu, odak geçiş kodunun, geçilecek
bir sonraki (veya ilk) odaklanabilir kontrolü bulmak için bir başlangıç
bağlamına sahip olmasını sağlar.

Bir odak kapsamı düğümünü odaklarsanız, önce alt ağacındaki geçerli veya
en son odaklanmış düğümü veya alt ağacında otomatik odaklanma
(autofocus) talep eden düğümü (varsa) odaklamaya çalışır. Böyle bir
düğüm yoksa, odağı kendisi alır.

## FocusableActionDetector Widget

`FocusableActionDetector`, eylemleri ve tuş bağlamalarını tanımlayan bir
dedektör oluşturmak ve odak ve üzerine gelme (hover) vurgularını işlemek
için geri çağrılar sağlamak amacıyla `Actions`, `Shortcuts`,
`MouseRegion` ve `Focus` widget’ının işlevselliğini birleştiren bir
widget’tır. Flutter kontrollerinin, kontrollerin tüm bu yönlerini
uygulamak için kullandığı şey budur. Sadece kurucu widget’lar
kullanılarak uygulandığından, tüm işlevlerine ihtiyacınız yoksa sadece
ihtiyaç duyduklarınızı kullanabilirsiniz, ancak bu davranışları özel
kontrollerinize entegre etmenin uygun bir yoludur.

[video](https://www.youtube.com/watch?v=R84AGg0lKs8)

## Odak Geçişini Kontrol Etmek

Bir uygulamanın odaklanma yeteneği olduğunda, birçok uygulamanın yapmak
istediği bir sonraki şey, kullanıcının klavyeyi veya başka bir giriş
cihazını kullanarak odağı kontrol etmesine izin vermektir. Bunun en
yaygın örneği, kullanıcının “bir sonraki” kontrole gitmek için `Tab`
tuşuna bastığı “tab geçişi”dir. “Bir sonraki”nin ne anlama geldiğini
kontrol etmek bu bölümün konusudur. Bu tür bir geçiş Flutter tarafından
varsayılan olarak sağlanır.

Basit bir ızgara düzeninde, hangi kontrolün bir sonraki olduğuna karar
vermek oldukça kolaydır. Sıranın sonunda değilseniz, sağdaki (veya
sağdan sola yerel ayarlar için soldaki) kontroldür. Bir sıranın
sonundaysanız, bir sonraki sıradaki ilk kontroldür. Ne yazık ki,
uygulamalar nadiren ızgaralar halinde düzenlenir, bu nedenle genellikle
daha fazla rehberliğe ihtiyaç duyulur.

Flutter’daki varsayılan algoritma (`ReadingOrderTraversalPolicy`), odak
geçişi için oldukça iyidir: Çoğu uygulama için doğru cevabı verir.
Ancak, her zaman patolojik durumlar veya bağlamın veya tasarımın
varsayılan sıralama algoritmasının ulaştığından farklı bir sıra
gerektirdiği durumlar vardır. Bu durumlar için, istenen sırayı elde
etmek için başka mekanizmalar vardır.

### FocusTraversalGroup Widget

`FocusTraversalGroup` widget’ı, başka bir widget’a veya widget grubuna
geçmeden önce tamamen gezilmesi gereken widget alt ağaçlarının etrafına
ağaca yerleştirilmelidir. Sadece widget’ları ilgili gruplar halinde
gruplamak genellikle birçok tab geçişi sıralama sorununu çözmek için
yeterlidir. Değilse, gruba grup içindeki sıralamayı belirlemek için bir
`FocusTraversalPolicy` verilebilir.

Varsayılan `ReadingOrderTraversalPolicy` genellikle yeterlidir, ancak
sıralama üzerinde daha fazla kontrole ihtiyaç duyulan durumlarda
`OrderedTraversalPolicy` kullanılabilir. Odaklanabilir bileşenlerin
etrafına sarılmış `FocusTraversalOrder` widget’ının `order` argümanı
sırayı belirler. Sıra, herhangi bir `FocusOrder` alt sınıfı olabilir,
ancak `NumericFocusOrder` ve `LexicalFocusOrder` sağlanmıştır.

Sağlanan odak geçiş politikalarının hiçbiri uygulamanız için yeterli
değilse, kendi politikanızı da yazabilir ve istediğiniz herhangi bir
özel sıralamayı belirlemek için kullanabilirsiniz.

İşte `NumericFocusOrder` kullanarak bir düğme satırını İKİ, BİR, ÜÇ
sırasıyla gezmek için `FocusTraversalOrder` widget’ının nasıl
kullanılacağına dair bir örnek.

``` dart
class OrderedButtonRow extends StatelessWidget {
  const OrderedButtonRow({super.key});

  @override
  Widget build(BuildContext context) {
    return FocusTraversalGroup(
      policy: OrderedTraversalPolicy(), // Sıralı politikayı kullan
      child: Row(
        children: <Widget>[
          const Spacer(),
          FocusTraversalOrder(
            order: const NumericFocusOrder(2),
            child: TextButton(child: const Text('BİR'), onPressed: () {}),
          ),
          const Spacer(),
          FocusTraversalOrder(
            order: const NumericFocusOrder(1),
            child: TextButton(child: const Text('İKİ'), onPressed: () {}),
          ),
          const Spacer(),
          FocusTraversalOrder(
            order: const NumericFocusOrder(3),
            child: TextButton(child: const Text('ÜÇ'), onPressed: () {}),
          ),
          const Spacer(),
        ],
      ),
    );
  }
}
```

### FocusTraversalPolicy

`FocusTraversalPolicy`, bir talep ve mevcut odak düğümü verildiğinde bir
sonraki widget’ın hangisi olduğunu belirleyen nesnedir. Talepler (üye
işlevler) `findFirstFocus`, `findLastFocus`, `next`, `previous` ve
`inDirection` gibi şeylerdir.

`FocusTraversalPolicy`, `ReadingOrderTraversalPolicy`,
`OrderedTraversalPolicy` ve `DirectionalFocusTraversalPolicyMixin`
sınıfları gibi somut politikalar için soyut temel sınıftır.

Bir `FocusTraversalPolicy` kullanmak için, politikanın etkili olacağı
widget alt ağacını belirleyen bir `FocusTraversalGroup`’a verirsiniz.
Sınıfın üye işlevleri nadiren doğrudan çağrılır: bunlar odak sistemi
tarafından kullanılmak üzere tasarlanmıştır.

## Odak Yöneticisi (Focus Manager)

`FocusManager`, sistem için mevcut birincil odağı korur. Odak sistemi
kullanıcıları için yararlı olan sadece birkaç API parçasına sahiptir.
Biri, o anda odaklanmış odak düğümünü içeren ve aynı zamanda global
`primaryFocus` alanından da erişilebilen
`FocusManager.instance.primaryFocus` özelliğidir.

Diğer yararlı özellikler `FocusManager.instance.highlightMode` ve
`FocusManager.instance.highlightStrategy`’dir. Bunlar, odak vurguları
için “dokunma” modu ve “geleneksel” (fare ve klavye) mod arasında geçiş
yapması gereken widget’lar tarafından kullanılır. Bir kullanıcı gezinmek
için dokunmayı kullandığında, odak vurgusu genellikle gizlenir ve bir
fareye veya klavyeye geçtiklerinde, neyin odaklandığını bilmeleri için
odak vurgusunun tekrar gösterilmesi gerekir. `highlightStrategy`, odak
yöneticisine cihazın kullanım modundaki değişiklikleri nasıl
yorumlayacağını söyler: en son giriş olaylarına bağlı olarak ikisi
arasında otomatik olarak geçiş yapabilir veya dokunma veya geleneksel
modlarda kilitlenebilir. Flutter’da sağlanan widget’lar bu bilgiyi nasıl
kullanacaklarını zaten bilirler, bu nedenle yalnızca sıfırdan kendi
kontrollerinizi yazıyorsanız buna ihtiyacınız vardır. Vurgulama
modundaki değişiklikleri dinlemek için `addHighlightModeListener` geri
çağrısını kullanabilirsiniz.

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/ui/interactivity/focus

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir